# Function Calling

In [ ]:
import pandas as pd
import numpy as np
import os
import openai
from openai import OpenAI
import json

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = '/content/drive/MyDrive/project6_2/'

In [ ]:
!pip install openai

## OpenAI API Key 환경 변수 설정

* 제공받은 open ai api key를 **api_key.txt** 파일에 저장합니다.
    * (제공받은 api_key.txt 파일은 비어 있습니다.)

* 다음 코드를 통해 환경변수로 등록 합니다.

In [ ]:
def load_file(filepath):
    with open(filepath, 'r') as file:
        return file.readline().strip()

# API 키 로드 및 환경변수 설정
openai.api_key = load_file(path + 'api_key.txt') # 키 로드
os.environ['OPENAI_API_KEY'] = openai.api_key    # 환경변수로 저장

- 음성 파일 변환 함수 + 문서 요약 함수

In [ ]:
def conversion_summarize(audio_path, filename):
    # OpenAI 클라이언트 생성
    client = OpenAI()

    # 오디오 파일을 읽어서, 위스퍼를 사용한 변환
    audio_file = open(audio_path + filename, "rb")
    transcript = client.audio.transcriptions.create(
        file=audio_file,
        model="whisper-1",
        language="ko",
        response_format="text"
    )

    # 변환된 텍스트
    input_text = transcript

    # 시스템 역할과 응답 형식 지정
    system_role = '''당신은 응급상황에 대한 텍스트에서 핵심 내용을 훌륭하게 요약해주는 어시스턴트입니다.
    응답은 다음의 형식을 지켜주세요.
    {"summary": \"텍스트 요약\",
    "keyword" : \"핵심 키워드(3가지)\"}
    '''

    # 입력데이터를 GPT-3.5-turbo에 전달하고 답변 받아오기
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {
                "role": "system",
                "content": system_role
            },
            {
                "role": "user",
                "content": input_text
            }
        ]
    )

    # 응답 받기
    answer = response.choices[0].message.content

    # 응답 형식을 JSON으로 정리하고 요약 반환하기
    answer_dict = json.loads(answer)
    return answer_dict['summary']

- 응급 등급 분류 함수

In [ ]:
def classify_emergency_level(summary):
  ########
  return

- 응급실 추천 함수

In [ ]:
def recommend_hospitals(emergency_level, caller_location, hospital_data):
  #############
  return

- Function Calling

In [ ]:
client = OpenAI()

response = client.chat.completions.create(
    model = "gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are a an assisteant for emergency response"},
        {"role":"user", "content": "응급 전화를 처리하고 가까운 응급실 3곳을 추천해주세요. 음성 파일 경로는 'emergency_call.mp3' 입니다."}
    ],
    functions = [
        {

            "name": "conversion_summarize",  # 음성파일변환+문서요약 함수
            "description": "Recognize speech from an audio file and summarize the emergency context.",     # 함수의 역할 설명
            "parameters": {                  # 매개변수: 함수 호출 시 전달된 매개변수의 형식과 제약 조건을 정의
                "type" : "object",
                "properties": {              # 입력 매개변수의 구조 정의
                    "audio_file_path": {"type": "string",
                                        "description": "The path to the audio file containing the emergency call."},
                },
                "required": ["audio_file_path"],
            },
        },
        {
            "name": "classify_emergency_level",  # 응급 등급 분류 함수
            "descriotion": "Classify the emergency level based on the provided emergency context.",
            "parameters": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string", "description": "Summarized emergency context"},
                },
                "required": ["summary"],
            },
        },
        {
            "name": "recommend_hospitals",    # 응급실 추천 함수
            "description": "Recommend the nearest hospitals based on emergency level, caller location, and hospital information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "emergency_level": {"type": "string", "description": "Classified emergency level"},
                    "caller_location": {"type": "string", "description": "Location of the emergency caller"},  # 응급 환자(발신자) 위치
                    "hospital_info": {"type": "array", "items": {"type": "object"}, "description": "List of available hospitals with their locations and capacity."},  # 가능한 병원 위치와 수용인원  --> 더 추가할 사항 있으면 수정
                },
                "required": ["emergency_level", "caller_location", "hospital_info"],
            },
        },
    ]
)

# 응답 처리
# message = response['choices'][0]['message']
message = response.choices[0].message

# Function Calling 확인
if "function_call" in message:                                  # gpt응답에 함수 호출 요청이 포함되었는지 확인
  function_name = message["function_call"]["name"]              # 호출할 함수 이름 가져오기
  arguments = json.loads(message["funcion_call"]["arguments"])  # 함수 호출 시 전달된 인수

  if function_name == "conversion_summarize":
    result = conversion_summarize(**arguments)
    print("Recognized and Summarized:", result)

    # 응급 등급 분류
    classify_result = classify_emergency_level(result["summary"])
    print("Emergency Level Classification:", classify_result)

    # 응급실 추천
    hospital_recommendations = recommend_hospitals(
        emergency_level=classify_result["label"],      # 응급 등급
        hospital_data=hospital_data  # 병원 데이터
    )
    print("Hospital Recommendations:", hospital_recommendations)